In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

# === Step 1: Load data ===
df = pd.read_csv("../Dashboard_Final/avg_gap_scores.csv")  # Replace with actual filename


# === Step 2: Select clustering features ===
features = ['gap_score', 'interaction_count', 'share_count']
df['share_count'] = df['share_count'] * 10  # Increase importance
X = df[features].copy()


# === Step 3: Scale features ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === Step 4: Choose best k using silhouette score ===
sil_scores = []
K_range = range(2, 10)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

# Plot silhouette scores
plt.figure(figsize=(6, 4))
plt.plot(K_range, sil_scores, marker='o')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Analysis for KMeans")
plt.grid(True)
plt.show()

# === Step 5: Use best k (or manually pick) ===
best_k = K_range[sil_scores.index(max(sil_scores))]
kmeans = KMeans(n_clusters=best_k, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)

# === Step 6: Interpret clusters ===
summary = df.groupby('cluster')[features].mean().round(3)
print("Cluster Summary:")
print(summary)



In [ ]:
# Create dictionary of cluster labels to zip code lists
zipcode_clusters = df.groupby('cluster')['zipcode'].apply(list).to_dict()

# Print zip codes per cluster
for label, zips in zipcode_clusters.items():
    print(f"\n{label} Zip Codes ({len(zips)}):")
    print(zips)


In [ ]:
cluster_means = df.groupby('cluster')['gap_score'].mean().sort_values()
print("Average gap_score by cluster:")
print(cluster_means)
